In [ ]:
# Load the required libraries
import logging
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import networkx as nx

### Loading the fabricated store delivery data

In [ ]:
delimiter = ','

key_cols = ["store", "delivery_date"]

expected_cols = ['store', 'country_store', 'dc', 'shipping_condition', 'doma',
       'delivery_date', 'monday_of_week', 'delivery_day_of_the_week',
       'delivery_schedule', 'sales_index', 'delivery_quantity']

default_rename_map = {"Monday_of_week":"monday_of_week"}

expected_dtypes = {
    "store": "string", "country_store": "string", "dc": "string",
    "shipping_condition": "string", "delivery_schedule": "string",
    "delivery_day_of_the_week": "int8", "sales_index": "float", "delivery_quantity": "int32",
}

df = pd.read_csv("../data/input/store_deliveries.csv", delimiter=delimiter)

In [ ]:
# defining preprocessing function for cleaning up the data

def preprocess_data(df_raw: pd.DataFrame,
                    expected_cols: list[str],
                    expected_dtypes: dict[str, str],
                    rename_map: dict[str, str],
                    utc: bool = True) -> pd.DataFrame:
    
    # rename columns
    df = df_raw.rename(columns=rename_map).copy()

    missing = [c for c in expected_cols if c not in df.columns]
    if missing:
        raise KeyError(f"missing columns: {missing}")
    df = df[expected_cols].drop_duplicates()

    # nulls would silently become 'nan' / True after astype, so stop here
    nulls = df.isna().sum()
    if nulls.any():
        raise ValueError(f"nulls found: {nulls[nulls > 0].to_dict()}")

    # check for same store + date
    if df.duplicated(["store", "delivery_date"]).any():
        raise ValueError("duplicate (store, delivery_date) rows with conflicting values")

    # doma: bool -> 1/0 (astype(bool) would turn the string 'False' into True)
        # doma: bool (True/False), robust to strings like 'True' / 'false' and to 1/0
    flag = df["doma"].map({True: True, False: False, "True": True, "False": False, "true": True, "false": False})
    if flag.isna().any():
        raise ValueError(f"unexpected doma values: {df.loc[flag.isna(), 'doma'].unique()[:5]}")
    df["doma"] = flag.astype(bool)

    # timestamps
    df["delivery_date"] = pd.to_datetime(df["delivery_date"], utc=utc)
    df["monday_of_week"] = pd.to_datetime(df["monday_of_week"], utc=utc)

    # remaining dtypes, e.g. {'store': 'string', 'delivery_day_of_the_week': 'int8', 'doma': 'int8'}
    df = df.astype(expected_dtypes)

    # the weekday and Monday must agree with delivery_date
    dow = df["delivery_date"].dt.day_of_week 
    if (df["delivery_day_of_the_week"] != dow).any():
        raise ValueError("delivery_day_of_the_week disagrees with delivery_date")
    if (df["monday_of_week"] != (df["delivery_date"] - pd.to_timedelta(dow, unit="D")).dt.normalize()).any():
        raise ValueError("monday_of_week disagrees with delivery_date")

    # basic time features
    d = df["delivery_date"]
    df["year"] = d.dt.year
    df["quarter"] = d.dt.quarter
    df["month"] = d.dt.month
    df["week_of_year"] = d.dt.isocalendar().week.astype("int8")
    df["day_of_month"] = d.dt.day
    df["is_weekend"] = (dow >= 5).astype("int8")

    return df.sort_values(["store", "delivery_date"]).reset_index(drop=True)

def plot_data(df, x=None, y=None, kind="bar", hue=None, agg="sum", sort=False, ascending=False,
              top_n=None, rotate=0, title=None, xlabel=None, ylabel=None, figsize=(10, 5), ax=None, **kwargs):
    """Quick plots from a DataFrame.

    x       column for the x axis; None = use the index (e.g. a groupby result)
    y       column name or list of columns (list = several bars/lines)
    kind    'bar' | 'barh' | 'line' | 'area' | 'scatter'
    hue     column to split into several bars/lines/colours (one y only)
    agg     how to combine rows that share the same x ('sum', 'mean', 'median', 'count', ... or None = no aggregation)
    sort    sort by the first y (bar kinds); top_n keeps only the first n after sorting
    rotate  rotation of x tick labels; extra kwargs go to pandas .plot / ax.scatter (color=, alpha=, stacked=True, ...)
    """
    ys = [y] if isinstance(y, str) else list(y)
    data = df.reset_index() if x is None else df
    x = data.columns[0] if x is None else x
    if ax is None:
        _, ax = plt.subplots(figsize=figsize)

    if kind == "scatter":
        for name, g in (data.groupby(hue, observed=True) if hue else [(None, data)]):
            ax.scatter(g[x], g[ys[0]], label=name, **{"s": 12, "alpha": 0.6, **kwargs})
        if hue:
            ax.legend(title=hue)
    else:
        if hue:
            plot_df = data.pivot_table(index=x, columns=hue, values=ys[0], aggfunc=agg or "sum", observed=True)
        elif agg:
            plot_df = data.groupby(x, observed=True)[ys].agg(agg)
        else:
            plot_df = data.set_index(x)[ys]
        if sort or top_n:
            plot_df = plot_df.sort_values(plot_df.columns[0], ascending=ascending)
        if top_n:
            plot_df = plot_df.head(top_n)
        plot_df.plot(kind=kind, ax=ax, **kwargs)

    value_label = ylabel or (ys[0] if len(ys) == 1 else "value")
    ax.set_title(title or f"{', '.join(ys)} by {x}" + (f" (per {hue})" if hue else ""))
    if kind == "barh":
        ax.set_xlabel(value_label); ax.set_ylabel(xlabel or x)
    else:
        ax.set_xlabel(xlabel or x); ax.set_ylabel(value_label)
        ax.tick_params(axis="x", rotation=rotate)
    ax.grid(alpha=0.3)
    ax.figure.tight_layout()
    return ax

In [ ]:
df = preprocess_data(df,
                expected_cols,
                expected_dtypes,
                default_rename_map)

### Basic counts of the data

In [ ]:
# country level stats
summary_country = (
    df.groupby("country_store")
      .agg(stores=("store", "nunique"),
           dcs=("dc", "nunique"),
           deliveries=("delivery_date", "size"),
           total_colli=("delivery_quantity", "sum"),
           avg_colli_per_delivery=("delivery_quantity", "mean"),
           doma_share=("doma", "mean"))
      .round(2)
      .sort_values("stores", ascending=False)
)
summary_country

In [ ]:
summary_store = (
    df.groupby("store")
      .agg(stores=("store", "nunique"),
           dcs=("dc", "nunique"),
           deliveries=("delivery_date", "size"),
           total_colli=("delivery_quantity", "sum"),
           avg_colli_per_delivery=("delivery_quantity", "mean"),
           doma_share=("doma", "mean"))
      .round(2)
      .sort_values("stores", ascending=False)
)
summary_store.head(20)

### Weekly delivery quantities

In [ ]:
plot_data(df, x="monday_of_week", y="delivery_quantity", kind="line", hue="country_store")

### Weekly total colli over time, with the Christmas weeks marked

In [ ]:
christmas_weeks_after = 4   # as in config.py: the Christmas week and the 4 weeks after it

delivery_dates = pd.to_datetime(df["delivery_date"], utc=True).dt.tz_localize(None)
df["week_start"] = delivery_dates.dt.normalize() - pd.to_timedelta(delivery_dates.dt.day_of_week, unit="D")
weekly_total = df.groupby("week_start")["delivery_quantity"].sum().sort_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(weekly_total.index, weekly_total.values / 1e3, color="#1f4e79", lw=1.8)

first, last = weekly_total.index.min(), weekly_total.index.max()
for year in range(first.year, last.year + 1):
    christmas = pd.Timestamp(year, 12, 25)
    christmas_monday = christmas - pd.Timedelta(days=christmas.day_of_week)    # Monday of the ISO week with 25 December
    if christmas_monday < first - pd.Timedelta(weeks=1) or christmas_monday > last:
        continue
    period_end = min(christmas_monday + pd.Timedelta(weeks=christmas_weeks_after + 1), last + pd.Timedelta(weeks=1))
    ax.axvspan(christmas_monday, period_end, color="#d9822b", alpha=0.18, lw=0)
    ax.axvline(christmas_monday, color="#b5481b", ls="--", lw=1.2)
    ax.annotate(f"Christmas week\n{christmas_monday:%d %b %Y}", (christmas_monday, weekly_total.max() / 1e3),
                xytext=(6, -4), textcoords="offset points", va="top", fontsize=9, color="#b5481b")

ax.plot([], [], color="#b5481b", ls="--", label="Christmas week (ISO week with 25 Dec)")
ax.fill_between([], [], color="#d9822b", alpha=0.18, label=f"Holiday period (+{christmas_weeks_after} weeks)")
ax.set_title("Weekly total colli delivered (all stores)")
ax.set_ylabel("Colli per week (thousands)")
ax.set_xlim(first, last + pd.Timedelta(weeks=1))
ax.set_ylim(0)
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="lower left", frameon=False)
plt.show()


### Mean delivery quantities per day of the week

In [ ]:
plot_data(df, x="delivery_day_of_the_week", y="delivery_quantity", agg="mean", color="#4C78A8")

### Weekly colli per store against the number of delivery days in that week

In [ ]:
# Weekly colli per store against the number of delivery days in that week
store_week = (df.groupby(["store", "week_start"])
                .agg(delivery_days=("delivery_date", "nunique"), weekly_colli=("delivery_quantity", "sum"))
                .reset_index())
by_days = store_week.groupby("delivery_days")["weekly_colli"].agg(mean="mean", store_weeks="size")

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(by_days.index, by_days["mean"], color="#4C78A8")
for days, row in by_days.iterrows():
    ax.annotate(f"{row['mean']:,.0f}\nn={int(row['store_weeks']):,}", (days, row["mean"]),
                xytext=(0, 4), textcoords="offset points", ha="center", fontsize=9)
ax.set_xticks(by_days.index)
ax.set_xlabel("Delivery days in the week (per store)")
ax.set_ylabel("Mean weekly colli per store")
ax.set_title("Weekly volume per store against number of delivery days\n(n = number of store-weeks)")
ax.set_ylim(0, by_days["mean"].max() * 1.18)
ax.spines[["top", "right"]].set_visible(False)
plt.show()


### Delivery quantity means and sales index mean

In [ ]:
plot_data(df, x="country_store", y=["delivery_quantity", "sales_index"], agg="mean")

### Seasonality (sales index by ISO week, one line per year)

In [ ]:
s = df.drop_duplicates(['store', 'monday_of_week']).assign(
    woy=lambda d: d['week_of_year'], yr=lambda d: d['monday_of_week'].dt.year)
s = s.groupby(['woy', 'yr'])['sales_index'].mean().unstack()
ax = s.plot(marker='.')
ax.set(title='Mean sales_index by week of year', xlabel='ISO week', ylabel='sales_index')
plt.show()

### Delivery v total colli

In [ ]:
plot_data(summary_store, x="deliveries", y="total_colli", kind="scatter")

### Delivery day of the week and shipping conditions

In [ ]:
plot_data(df, x="delivery_day_of_the_week", y="delivery_quantity", hue="shipping_condition",
          agg="mean", kind="bar", stacked=True)

### Priority (`doma`) deliveries
Compared with the same store + weekday average, so the effect is not mixed up with store size or weekday.

In [ ]:
# `cover_days` looks at the store's next delivery date

df['cover_days'] = (df.groupby('store')['delivery_date'].shift(-1) - df['delivery_date']).dt.days.clip(upper=7)

base = df.groupby(['store', 'delivery_day_of_the_week', 'cover_days'])['delivery_quantity'].transform('median')
df['vs_typical'] = df['delivery_quantity'] / base
d = df.dropna(subset=['cover_days'])
fig, ax = plt.subplots(figsize=(5, 4))
ax.boxplot([d.loc[d['doma'] == 0, 'vs_typical'], d.loc[d['doma'] == 1, 'vs_typical']],
           tick_labels=['regular', 'doma'], showfliers=False)
ax.set(title='Delivery size vs typical (same store, weekday, cover)', ylabel='ratio')
print(d.groupby('doma')['vs_typical'].median().round(3).to_dict())
plt.show()

### Many to many

In [ ]:
edges = df[['dc', 'store']].drop_duplicates()
G = nx.from_pandas_edgelist(edges, source='dc', target='store')

dcs = edges['dc'].unique()
palette = plt.cm.tab20.colors  # tab10 has only 10 colors; tab20 gives 20
dc_color = {dc: palette[i % len(palette)] for i, dc in enumerate(dcs)}


store_to_dc = edges.drop_duplicates('store').set_index('store')['dc']

node_colors = [dc_color[n] if n in dc_color else dc_color[store_to_dc[n]] for n in G]
node_sizes  = [900 if n in dc_color else 40 for n in G]

plt.figure(figsize=(15,15))
pos = nx.spring_layout(G, seed=9, k=0.2)
nx.draw(G, pos, node_color=node_colors, node_size=node_sizes,
        edge_color='lightgrey', with_labels=False)
nx.draw_networkx_labels(G, pos, labels={n: n for n in dcs}, font_size=9)
plt.show()

### Checking for when stores first appear

In [ ]:
def store_lifespan(df, store="store", date="delivery_date", inactive_after_days=28):
    """First and last delivery per store, plus how long it was active.

    A store counts as 'active' if its last delivery is within `inactive_after_days`
    of the latest date in the data (so stores that were closed or dropped show up).
    """
    d = pd.to_datetime(df[date])
    out = (df.assign(_d=d)
             .groupby(store)["_d"]
             .agg(first_delivery="min", last_delivery="max", n_deliveries="size"))
    out["days_active"] = (out["last_delivery"] - out["first_delivery"]).dt.days + 1
    out["weeks_active"] = (out["days_active"] / 7).round(1)
    out["started_late"] = out["first_delivery"] > d.min() + pd.Timedelta(days=7)    # opened after the data starts
    out["is_active"] = out["last_delivery"] >= d.max() - pd.Timedelta(days=inactive_after_days)
    return out.sort_values("first_delivery")

In [ ]:
life = store_lifespan(df)
life.head()

In [ ]:
 # stores that stopped receiving deliveries

life[~life.is_active] 

In [ ]:
# stores that appear later

life[life.started_late]